# DeepEval — Pytest-Style LLM Testing

**DeepEval** brings **pytest-style assertions** to LLM and agent outputs. Instead of `assert response == "hello"`, you write `assert_test(test_case, metrics)` where each **metric** scores a dimension (relevancy, faithfulness, policy compliance) against a **threshold**.

```
  pytest test_agent.py
       |
  test_refund_reply()
       |
  LLMTestCase(input, actual_output, context)
       |
  +----------+----------+
  | Metric 1 | Metric 2 |  threshold >= 0.7
  +----------+----------+
       |
  PASS / FAIL + score report
```

This notebook implements a **MiniDeepEval** framework in plain Python for **ShopCo Support Hub** and **ReleaseFlow** — test cases, built-in metrics, custom metrics, parametrized suites, and a pytest-like runner.

In [ ]:
"""
DeepEval-style pytest LLM testing — ShopCo + ReleaseFlow.
"""

import json
import re
import traceback
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

USE_LIVE_LLM = False
DEFAULT_THRESHOLD = 0.7


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "customer": "alice@shop.com", "status": "shipped", "total_usd": 1299.0},
    "ORD-1002": {"order_id": "ORD-1002", "customer": "bob@shop.com", "status": "processing", "total_usd": 449.0},
}

POLICIES: dict[str, str] = {
    "returns": "30-day returns on unused items. [pol-returns-01]",
    "refunds": "Cite policy before promising refunds. Escalate over $1000. [pol-refunds-03]",
}

RELEASE_CHECKS: dict[str, dict[str, str]] = {
    "unit_tests": {"status": "pass", "detail": "412 tests passed"},
    "security_scan": {"status": "warn", "detail": "1 medium CVE in dev dependency"},
    "lint": {"status": "pass", "detail": "no issues"},
}

TEST_RUN_LOG: list[dict[str, Any]] = []

print("MiniDeepEval lab ready")

---

## 1. Unit Tests vs LLM Tests

| Traditional pytest | DeepEval-style |
|--------------------|----------------|
| `assert fn(x) == 42` | `assert_test(case, metrics)` |
| Deterministic | Probabilistic / scored |
| Exact match | Threshold on metric (0.0–1.0) |
| Fast | May call judge model |

LLM tests belong in **CI eval suites** and **pre-release gates**, not necessarily on every keystroke.

---

## 2. LLMTestCase — Input, Output, Context

In [ ]:
@dataclass
class LLMTestCase:
    """Mirrors DeepEval LLMTestCase — holds everything a metric needs."""
    input: str
    actual_output: str
    expected_output: str | None = None
    context: list[str] = field(default_factory=list)
    retrieval_context: list[str] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)


refund_case = LLMTestCase(
    input="I want a refund for ORD-1001",
    actual_output=(
        "We understand your frustration. ORD-1001 is shipped ($1299). "
        "Per policy: [pol-refunds-03]. A specialist will follow up."
    ),
    expected_output="Refund guidance with policy citation and escalation for high-value order.",
    context=[POLICIES["refunds"], f"Order: {ORDERS['ORD-1001']}"],
    retrieval_context=[POLICIES["refunds"]],
    metadata={"domain": "shopco", "intent": "refund"},
)

print(f"Test case: {refund_case.input[:40]}...")

---

## 3. Metric Base Class and Results

In [ ]:
@dataclass
class MetricResult:
    metric_name: str
    score: float
    passed: bool
    threshold: float
    reason: str


class BaseMetric:
    """Base metric — subclasses implement measure()."""

    def __init__(self, threshold: float = DEFAULT_THRESHOLD):
        self.threshold = threshold
        self.name = self.__class__.__name__

    def measure(self, test_case: LLMTestCase) -> MetricResult:
        raise NotImplementedError


print("Metric base ready")

---

## 4. Built-in Metrics — Offline Implementations

In [ ]:
class AnswerRelevancyMetric(BaseMetric):
    """Does the output address the user input?"""

    def measure(self, test_case: LLMTestCase) -> MetricResult:
        inp = test_case.input.lower()
        out = test_case.actual_output.lower()
        hits = 0
        checks = 0
        if "refund" in inp:
            checks += 1
            if "refund" in out or "policy" in out or "pol-" in out:
                hits += 1
        oid = re.search(r"ORD-\d+", test_case.input.upper())
        if oid:
            checks += 1
            if oid.group().lower() in out:
                hits += 1
        score = hits / checks if checks else (1.0 if out else 0.0)
        return MetricResult(self.name, score, score >= self.threshold, self.threshold, "Relevancy to input topics.")


class FaithfulnessMetric(BaseMetric):
    """Is the output grounded in provided context / retrieval?"""

    def measure(self, test_case: LLMTestCase) -> MetricResult:
        ctx = " ".join(test_case.context + test_case.retrieval_context).lower()
        out = test_case.actual_output.lower()
        if not ctx:
            return MetricResult(self.name, 1.0, True, self.threshold, "No context to verify.")
        anchors = ["pol-refunds", "shipped", "1299", "processing", "449"]
        grounded = sum(1 for a in anchors if a in ctx and a in out)
        claimed = sum(1 for a in anchors if a in out)
        score = grounded / claimed if claimed else 1.0
        return MetricResult(self.name, score, score >= self.threshold, self.threshold, f"Grounded {grounded}/{claimed} claims.")


class PolicyComplianceMetric(BaseMetric):
    """Refund replies must cite policy ID."""

    def measure(self, test_case: LLMTestCase) -> MetricResult:
        is_refund = "refund" in test_case.input.lower()
        has_citation = "[pol-" in test_case.actual_output or "pol-refunds" in test_case.actual_output.lower()
        score = 1.0 if (not is_refund or has_citation) else 0.0
        return MetricResult(self.name, score, score >= self.threshold, self.threshold, "Policy citation present." if has_citation else "Missing citation.")


class HallucinationMetric(BaseMetric):
    """Penalize facts not supported by context."""

    def measure(self, test_case: LLMTestCase) -> MetricResult:
        ctx = " ".join(test_case.context).lower()
        out = test_case.actual_output.lower()
        hallucination = "ord-9999" in out and "ord-9999" not in ctx
        score = 0.0 if hallucination else 1.0
        return MetricResult(self.name, score, score >= self.threshold, self.threshold, "Hallucination detected." if hallucination else "No hallucination.")


for m in (AnswerRelevancyMetric(), FaithfulnessMetric(), PolicyComplianceMetric()):
    r = m.measure(refund_case)
    print(f"{r.metric_name}: {r.score:.2f} pass={r.passed}")

---

## 5. assert_test — The Core DeepEval Pattern

In [ ]:
class AssertTestError(AssertionError):
    def __init__(self, test_case: LLMTestCase, results: list[MetricResult]):
        failed = [r for r in results if not r.passed]
        msg = "; ".join(f"{r.metric_name}={r.score:.2f} (need {r.threshold})" for r in failed)
        super().__init__(msg)
        self.results = results


@dataclass
class TestResult:
    test_name: str
    passed: bool
    metric_results: list[MetricResult]
    error: str | None = None


def assert_test(test_case: LLMTestCase, metrics: list[BaseMetric]) -> list[MetricResult]:
    """DeepEval-style: run all metrics; raise if any fail threshold."""
    results = [m.measure(test_case) for m in metrics]
    if not all(r.passed for r in results):
        raise AssertTestError(test_case, results)
    return results


def safe_assert_test(name: str, test_case: LLMTestCase, metrics: list[BaseMetric]) -> TestResult:
    try:
        results = assert_test(test_case, metrics)
        return TestResult(name, True, results)
    except AssertTestError as e:
        return TestResult(name, False, e.results, str(e))


def expect_metrics_fail(name: str, test_case: LLMTestCase, metrics: list[BaseMetric]) -> TestResult:
    """Negative test: passes when metrics correctly reject bad output."""
    tr = safe_assert_test(name, test_case, metrics)
    if tr.passed:
        return TestResult(name, False, tr.metric_results, "Expected metrics to fail but they passed")
    return TestResult(name, True, tr.metric_results)


shopco_metrics = [AnswerRelevancyMetric(), FaithfulnessMetric(), PolicyComplianceMetric()]
tr = safe_assert_test("test_good_refund", refund_case, shopco_metrics)
print(f"{tr.test_name}: pass={tr.passed}")

---

## 6. Pytest-Style Test Functions — ShopCo

In [ ]:
def test_refund_reply_includes_policy() -> TestResult:
    case = LLMTestCase(
        input="Refund for ORD-1001 please",
        actual_output=refund_case.actual_output,
        context=refund_case.context,
        retrieval_context=refund_case.retrieval_context,
    )
    return safe_assert_test("test_refund_reply_includes_policy", case, shopco_metrics)


def test_bad_refund_reply_fails() -> TestResult:
    case = LLMTestCase(
        input="Refund for ORD-1001",
        actual_output="Sure, approved! Money back tomorrow.",
        context=[POLICIES["refunds"]],
    )
    return expect_metrics_fail("test_bad_refund_reply_fails", case, shopco_metrics)


def test_order_status_reply() -> TestResult:
    case = LLMTestCase(
        input="Where is ORD-1002?",
        actual_output="ORD-1002 is currently processing.",
        context=[str(ORDERS["ORD-1002"])],
    )
    return safe_assert_test("test_order_status_reply", case, [AnswerRelevancyMetric(), HallucinationMetric()])


for fn in (test_refund_reply_includes_policy, test_bad_refund_reply_fails, test_order_status_reply):
    r = fn()
    print(f"{r.test_name}: {'PASS' if r.passed else 'FAIL'} {r.error or ''}")

---

## 7. Custom Metric — Escalation for High-Value Orders

In [ ]:
class EscalationMetric(BaseMetric):
    """Custom metric: refunds over $1000 need escalation mention."""

    def measure(self, test_case: LLMTestCase) -> MetricResult:
        oid_match = re.search(r"ORD-\d+", test_case.input.upper())
        if not oid_match or "refund" not in test_case.input.lower():
            return MetricResult(self.name, 1.0, True, self.threshold, "Not applicable.")
        oid = oid_match.group()
        order = ORDERS.get(oid)
        if not order or order["total_usd"] <= 1000:
            return MetricResult(self.name, 1.0, True, self.threshold, "Below escalation threshold.")
        has_esc = any(w in test_case.actual_output.lower() for w in ("specialist", "follow up", "escalat"))
        score = 1.0 if has_esc else 0.0
        return MetricResult(self.name, score, score >= self.threshold, self.threshold, "Escalation mentioned." if has_esc else "Missing escalation.")


esc_result = EscalationMetric().measure(refund_case)
print(f"EscalationMetric: {esc_result.score} pass={esc_result.passed}")

---

## 8. ReleaseFlow Test Suite

In [ ]:
class ReleaseDecisionMetric(BaseMetric):
    def measure(self, test_case: LLMTestCase) -> MetricResult:
        out = test_case.actual_output.lower()
        sec_warn = RELEASE_CHECKS["security_scan"]["status"] == "warn"
        if sec_warn:
            correct = "no-go" in out and ("security" in out or "cve" in out)
        else:
            correct = "go" in out
        score = 1.0 if correct else 0.0
        return MetricResult(self.name, score, score >= self.threshold, self.threshold, "Correct ship decision." if correct else "Wrong decision.")


class ReleaseCompletenessMetric(BaseMetric):
    def measure(self, test_case: LLMTestCase) -> MetricResult:
        out = test_case.actual_output.lower()
        parts = sum(["test" in out, "lint" in out, "security" in out or "cve" in out])
        score = parts / 3
        return MetricResult(self.name, score, score >= self.threshold, self.threshold, f"Covers {parts}/3 check types.")


def test_release_nogo_on_security() -> TestResult:
    case = LLMTestCase(
        input="Ship v2.4.1?",
        actual_output="NO-GO: security scan warn — 1 medium CVE. Tests: 412 passed. Lint: pass.",
        context=[pretty(RELEASE_CHECKS)],
    )
    metrics = [ReleaseDecisionMetric(), ReleaseCompletenessMetric()]
    return safe_assert_test("test_release_nogo_on_security", case, metrics)


def test_release_bad_summary_fails() -> TestResult:
    case = LLMTestCase(
        input="Ship v2.4.1?",
        actual_output="GO: ship it now!",
        context=[pretty(RELEASE_CHECKS)],
    )
    return expect_metrics_fail("test_release_bad_summary_fails", case, [ReleaseDecisionMetric()])


for fn in (test_release_nogo_on_security, test_release_bad_summary_fails):
    r = fn()
    print(f"{r.test_name}: {'PASS' if r.passed else 'FAIL'}")

---

## 9. Parametrized Tests

In [ ]:
PARAMETRIZED_CASES = [
    ("ord1002_status", LLMTestCase("ORD-1002 status?", "ORD-1002 is processing.", context=[str(ORDERS["ORD-1002"])]), True),
    ("ord1001_refund_good", refund_case, True),
    ("ord1001_refund_bad", LLMTestCase("Refund ORD-1001", "Approved instantly.", context=[POLICIES["refunds"]]), False),
]


def run_parametrized(cases: list[tuple[str, LLMTestCase, bool]], metrics: list[BaseMetric]) -> list[TestResult]:
    results = []
    for name, case, expect_pass in cases:
        tr = safe_assert_test(f"test_param_{name}", case, metrics)
        metric_passed = tr.passed
        meta_ok = metric_passed == expect_pass
        results.append(TestResult(
            f"test_param_{name}", meta_ok, tr.metric_results,
            None if meta_ok else f"Expected metric_pass={expect_pass}, got {metric_passed}",
        ))
    return results


param_results = run_parametrized(PARAMETRIZED_CASES, shopco_metrics)
for r in param_results:
    print(f"{r.test_name}: {'PASS' if r.passed else 'FAIL'}")

---

## 10. Test Runner — Collect and Execute Suite

In [ ]:
class TestRunner:
    """Minimal pytest-like collector and runner."""

    def __init__(self):
        self.tests: list[tuple[str, Callable[[], TestResult]]] = []

    def register(self, fn: Callable[[], TestResult]) -> None:
        self.tests.append((fn.__name__, fn))

    def run_all(self) -> dict[str, Any]:
        results: list[TestResult] = []
        for name, fn in self.tests:
            try:
                results.append(fn())
            except Exception as e:
                results.append(TestResult(name, False, [], f"{type(e).__name__}: {e}"))
        passed = sum(1 for r in results if r.passed)
        report = {
            "total": len(results),
            "passed": passed,
            "failed": len(results) - passed,
            "pass_rate": passed / len(results) if results else 0,
            "results": [{"name": r.test_name, "passed": r.passed, "error": r.error} for r in results],
            "ts": utcnow(),
        }
        TEST_RUN_LOG.append(report)
        return report


runner = TestRunner()
for fn in [
    test_refund_reply_includes_policy,
    test_bad_refund_reply_fails,
    test_order_status_reply,
    test_release_nogo_on_security,
    test_release_bad_summary_fails,
]:
    runner.register(fn)

report = runner.run_all()
print(f"Suite: {report['passed']}/{report['total']} passed ({report['pass_rate']:.0%})")

---

## 11. Fixtures — Shared Agent and Context

In [ ]:
@dataclass
class ShopCoFixtures:
    """Pytest fixtures as a dataclass for notebook clarity."""
    refund_context: list[str]
    status_context: list[str]
    default_metrics: list[BaseMetric]


def shopco_fixtures() -> ShopCoFixtures:
    return ShopCoFixtures(
        refund_context=[POLICIES["refunds"], str(ORDERS["ORD-1001"])],
        status_context=[str(ORDERS["ORD-1002"])],
        default_metrics=[AnswerRelevancyMetric(), FaithfulnessMetric(), PolicyComplianceMetric(), EscalationMetric()],
    )


def test_refund_with_fixtures() -> TestResult:
    fx = shopco_fixtures()
    case = LLMTestCase(
        input="Refund ORD-1001",
        actual_output=refund_case.actual_output,
        context=fx.refund_context,
        retrieval_context=[POLICIES["refunds"]],
    )
    return safe_assert_test("test_refund_with_fixtures", case, fx.default_metrics)


fx_result = test_refund_with_fixtures()
print(f"{fx_result.test_name}: pass={fx_result.passed}")

---

## 12. evaluate() — Batch Metric Run (DeepEval API Shape)

In [ ]:
@dataclass
class EvaluateResult:
    test_case_index: int
    metric_results: list[MetricResult]
    all_passed: bool


def evaluate(test_cases: list[LLMTestCase], metrics: list[BaseMetric]) -> list[EvaluateResult]:
    """Batch evaluate — mirrors DeepEval evaluate() for datasets."""
    out: list[EvaluateResult] = []
    for i, case in enumerate(test_cases):
        results = [m.measure(case) for m in metrics]
        out.append(EvaluateResult(i, results, all(r.passed for r in results)))
    return out


dataset = [c for _, c, _ in PARAMETRIZED_CASES]
eval_results = evaluate(dataset, shopco_metrics)
for er in eval_results:
    scores = ", ".join(f"{r.metric_name}={r.score:.2f}" for r in er.metric_results)
    print(f"Case {er.test_case_index}: pass={er.all_passed} ({scores})")

---

## 13. CI Report Format

In [ ]:
def format_ci_report(report: dict[str, Any]) -> str:
    lines = [
        "=" * 50,
        f"MiniDeepEval CI Report — {report['ts']}",
        f"Passed: {report['passed']}/{report['total']} ({report['pass_rate']:.0%})",
        "=" * 50,
    ]
    for r in report["results"]:
        status = "PASS" if r["passed"] else "FAIL"
        lines.append(f"  [{status}] {r['name']}")
        if r.get("error"):
            lines.append(f"         {r['error']}")
    return "\n".join(lines)


print(format_ci_report(report))

---

## 14. Mapping to Real DeepEval

| MiniDeepEval (this notebook) | DeepEval library |
|------------------------------|------------------|
| `LLMTestCase` | `LLMTestCase` |
| `assert_test(case, metrics)` | `assert_test(test_case, metrics)` |
| `evaluate(cases, metrics)` | `evaluate(test_cases, metrics)` |
| `BaseMetric.measure()` | `BaseMetric.measure()` / `@metric` |
| `TestRunner` | `pytest` + `deepeval` plugin |

The **API shape** transfers directly; swap offline metrics for DeepEval's built-in or G-Eval judges when ready.

---

## 15. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| One metric only | Misses other failure modes | Bundle relevancy + faithfulness + domain metrics |
| Threshold too low | False passes | Calibrate on anchor set |
| No negative tests | Regressions undetected | Include `expect fail` cases |
| Flaky live judge in CI | Non-deterministic builds | Offline rules + nightly live eval |
| Giant monolithic test file | Hard to maintain | Parametrize + fixtures per domain |
| Ignoring metric reason | Cannot debug failures | Log score + reason per metric |

---

## 16. Check Your Understanding

1. How is `assert_test` different from a plain `assert`?
2. What fields does `LLMTestCase` carry?
3. Why separate **AnswerRelevancy** from **Faithfulness**?
4. When would you write a **custom metric**?
5. What is the purpose of **parametrized** test cases?
6. Why include tests that **expect failure**?
7. How does `evaluate()` differ from running tests one-by-one?

---

## 17. Optional — Live G-Eval Metric

Set `USE_LIVE_LLM = True` to score a test case with a live judge instead of offline rules.

In [ ]:
if USE_LIVE_LLM:
    import os
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Score 0-1 relevancy. Reply JSON {score, reason}."},
            {"role": "user", "content": f"Input: {refund_case.input}\nOutput: {refund_case.actual_output}"},
        ],
        response_format={"type": "json_object"},
    )
    print(resp.choices[0].message.content)
else:
    print("Offline mode — MiniDeepEval pytest-style tests for ShopCo and ReleaseFlow.")

---

## 18. Summary

DeepEval-style testing brings **scored assertions** to LLM outputs:

- **LLMTestCase** — input, output, context, retrieval context
- **Metrics** — relevancy, faithfulness, policy compliance, custom domain checks
- **assert_test** — fail CI when any metric is below threshold
- **Parametrize + fixtures** — scale test suites cleanly
- **evaluate()** — batch datasets for regression benchmarks

Start with offline metrics in CI; add live G-Eval judges in nightly runs. Keep negative tests to prove your metrics actually catch bad outputs.